In [ ]:
!pip install torchsummary
!pip install torchinfo

# Model with picobackbone

In [ ]:
import torch
from torchinfo import summary
from nets import pico


model = pico.yolo_v8_p(80).cuda()

# Create a dummy input tensor
dummy_input = torch.randn(1, 3, 640, 640).cuda()

# Display the model summary
summary(model, input_size=(1, 3, 640, 640), device='cuda')

## Model with darknet


In [ ]:
import torch
from torchsummary import summary
from nets import darknet

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = darknet.yolo_v8_n()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))


## Tiny Yolo

In [ ]:
import torch
from torchinfo import summary
from nets import tiny

class ModelWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tiny.YOLOTinyFPGA(num_classes=80)
    
    def forward(self, x):
        # Force training mode output
        self.model.train()
        return self.model(x)

# Create wrapped model and move to GPU
model = ModelWrapper().cuda()

# Display model summary with torchinfo
summary(model, 
        input_size=(1, 3, 640, 640),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        depth=4,
        device="cuda")

## Tinysimo Yolo

In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = tinysimo.yolo_v8()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))


In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = tinysimo.yolo_v8_s()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))

In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = tinysimo.yolo_v8_es()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))

In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo
import sys
from contextlib import redirect_stdout

class BackboneWrapper(torch.nn.Module):
             def __init__(self):
                          super().__init__()
                          # Create YOLO v8 nano model and get its backbone
                          self.model = tinysimo.yolo_v8_s()
                          self.backbone = self.model.net
             
             def forward(self, x):
                          # Return all feature maps from backbone
                          return self.backbone(x)

# Create and move model to GPU
model = BackboneWrapper().cuda()

# Save model summary to a text file
with open('model_summary.txt', 'w') as f:
             with redirect_stdout(f):
                          summary(model, (3, 640, 640))

In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo
import sys
from contextlib import redirect_stdout

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Instantiate YOLO model
        self.model = tinysimo.yolo_v8_s()
        
        # Properly wrap backbone layers using Sequential
        self.backbone = torch.nn.Sequential(*self.model.net.backbone)

    def forward(self, x):
        return self.backbone(x)

# Instantiate and summarize model
model = BackboneWrapper().cuda()

with open('model_summary.txt', 'w') as f:
    with redirect_stdout(f):
        summary(model, (3, 640, 640))


In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo
from contextlib import redirect_stdout

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Get the YOLO model
        model = tinysimo.yolo_v8_s()

        # Explicitly flatten and extract the backbone layers (Conv, Pool, etc.)
        modules = []
        for layer in model.net.backbone:
            if isinstance(layer, tinysimo.Conv):
                # Expand each Conv block explicitly
                modules.append(layer.conv)
                modules.append(layer.norm)
                modules.append(layer.relu)
            else:
                # For MaxPool2d layers
                modules.append(layer)

        # Now backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)

# Create and move model to GPU
model = BackboneWrapper().cuda()

# Save model summary to a text file
with open('model_summary.txt', 'w') as f:
    with redirect_stdout(f):
        summary(model, (3, 640, 640))


In [1]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimo

    # Get the YOLO model
    model = tinysimo.yolo_v8_s()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


In [ ]:
import torch
from torchsummary import summary
from nets import tinysimo

model = tinysimo.yolo_v8_s()

In [ ]:
print(model)